# Tuning and model selection

`align` decides how much the model relies on the anchor, and its useful value
depends on the data. It is the parameter worth tuning first.

In [1]:
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split

from lairnet import LAIRNetRegressor
from lairnet.model_selection import LAIRNetCV, default_grid

X, y = make_friedman1(n_samples=500, noise=1.0, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

default_grid()

{'align': [0.0, 0.1, 0.25, 0.5, 0.75, 0.9],
 'ridge': [0.0001, 0.001, 0.01, 0.1],
 'anchor_alpha': [0.0001, 0.01, 1.0]}

In [2]:
search = LAIRNetCV(
    LAIRNetRegressor(n_hidden=40, n_layers=8, random_state=0),
    param_grid={"align": [0.0, 0.25, 0.5, 0.75], "ridge": [1e-3, 1e-1]},
    cv=3,
).fit(X_train, y_train)

print("best params :", search.best_params_)
print("cv score    :", f"{search.best_score_:.4f}")
print("test score  :", f"{search.score(X_test, y_test):.4f}")

best params : {'align': 0.25, 'ridge': 0.1}
cv score    : 0.8153
test score  : 0.8463


## The alignment profile falls out for free

The best score achievable at each `align`, after tuning the other searched
parameters. No second sweep needed.

In [3]:
values, scores = search.align_profile()
for a, s in zip(values, scores):
    print(f"align={a:<5} {s:.4f}")

align=0.0   0.7044
align=0.25  0.8153
align=0.5   0.7999
align=0.75  0.7761


## The search result is usable directly

Prediction and the LAIR-specific inspection methods delegate to the selected
model, so you do not have to reach into `best_estimator_`.

In [4]:
print("layer predictions :", search.layer_predictions(X_test).shape)
print("anchor converged  :", search.anchor_converged_)
print("worst conditioning:", f"{search.layer_conditions_.max():.3g}")

layer predictions : (8, 150, 1)
anchor converged  : True
worst conditioning: 1.93e+05


## Classification works the same way

In [5]:
from sklearn.datasets import load_wine
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from lairnet import LAIRNetClassifier

Xc, yc = load_wine(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.3, random_state=0, stratify=yc)

clf = make_pipeline(
    StandardScaler(),
    LAIRNetClassifier(n_hidden=40, n_layers=6, random_state=0),
).fit(Xc_tr, yc_tr)

print("accuracy :", f"{clf.score(Xc_te, yc_te):.4f}")
print("classes  :", clf[-1].classes_)

accuracy : 1.0000
classes  : [0 1 2]


```{note}
`predict_proba` is a softmax over scores fitted by least squares, not a
likelihood. The ordering is meaningful; calibrate explicitly if you need the
magnitudes to be.
```